In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import random

In [ ]:
"""
1. 使用梯度检查点技术（以计算量为代价来节省显存的技术，在标准反向传播中，模型会保存所有中间激活值用于计算梯度，
   而使用梯度检查点技术，模型会保存所有中间激活值用于计算梯度，但是会减少显存消耗）
示例如下：
"""
import torch
import torch.nn as nn
import torch.utils.checkpoint as cp

class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(10, 20),
            nn.ReLU(),
            nn.Linear(20, 30),
            nn.ReLU()
        )
        self.decoder = nn.Linear(30, 10)

    def forward(self, x):
        if self.training:
            # 使用梯度检查点
            x = cp.checkpoint(self.encoder, x)
        else:
            # 正常前向传播
            x = self.encoder(x)
        return self.decoder(x)

model = MyModel()
input_data = torch.randn(1, 10)
output = model(input_data)

"""
在上述例子中，使用梯度检查点（Gradient Checkpointing）技术时，主要节省的是中间激活值（intermediate activations）的显存占用。具体来说，以下变量的显存会被节约：
1. 中间激活值
在标准的前向传播中，每一层的输出（即中间激活值）都会被保存下来，以便在反向传播时使用。这些中间激活值通常会占用大量的显存。例如，在 self.encoder 中的每一层（如 nn.Linear 和 nn.ReLU）的输出都会被保存。
2. 梯度检查点的机制
当使用 torch.utils.checkpoint.checkpoint 时，这些中间激活值不会被保存。相反，它们会在反向传播时重新计算。具体来说：
前向传播：torch.utils.checkpoint.checkpoint 会运行 self.encoder 的前向传播，但不会保存中间激活值。
反向传播：在反向传播时，torch.utils.checkpoint.checkpoint 会重新运行 self.encoder 的前向传播，以重新计算这些中间激活值，然后计算梯度。
3. 节省的显存
假设 self.encoder 包含以下层：
第一层：nn.Linear(10, 20)，输出形状为 (batch_size, 20)
第二层：nn.ReLU()，输出形状为 (batch_size, 20)
第三层：nn.Linear(20, 30)，输出形状为 (batch_size, 30)
第四层：nn.ReLU()，输出形状为 (batch_size, 30)
在标准的前向传播中，这些中间激活值都会被保存，占用的显存为：
第一层输出：batch_size * 20 * sizeof(float)
第二层输出：batch_size * 20 * sizeof(float)
第三层输出：batch_size * 30 * sizeof(float)
第四层输出：batch_size * 30 * sizeof(float)
使用梯度检查点后，这些中间激活值不会被保存，因此节省的显存为：
第一层输出：batch_size * 20 * sizeof(float)
第二层输出：batch_size * 20 * sizeof(float)
第三层输出：batch_size * 30 * sizeof(float)
第四层输出：batch_size * 30 * sizeof(float)
4. 计算开销
虽然显存被节省了，但计算开销会增加。因为反向传播时需要重新计算这些中间激活值，这会增加额外的计算时间。
总结
使用梯度检查点技术时，主要节省的是中间激活值的显存占用。这些中间激活值在前向传播中不会被保存，而是在反向传播时重新计算。虽然这会增加计算开销，但在显存受限的情况下，这是一个有效的权衡方法。
"""







In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
def setup_seed(seed):
    # 1. 设置pytorch的随机种子
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    # 2. 设置其他第3方库的随机变量
    np.random.seed(seed)
    random.seed(seed)
setup_seed(10101)

cpu


<span style='color:red;'>1.定义神经网络 </span>

In [3]:
class net(nn.Module):
    def __init__(self):
        super(net,self).__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(3*28*28,512),
            nn.ReLU(),
            nn.Linear(512,512),
            nn.ReLU(),
            nn.Linear(512,10)
        )
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
model = net().to(device)
print('model',model)

model net(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=2352, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [7]:
for name, i in model.named_parameters():
    print(name, i.shape)

linear_relu_stack.0.weight torch.Size([512, 2352])
linear_relu_stack.0.bias torch.Size([512])
linear_relu_stack.2.weight torch.Size([512, 512])
linear_relu_stack.2.bias torch.Size([512])
linear_relu_stack.4.weight torch.Size([10, 512])
linear_relu_stack.4.bias torch.Size([10])


In [27]:
# print([ x for x in model.modules()])

In [24]:
m = [ x for x in model.modules()]
print(m[1]) # Flatten(start_dim=1, end_dim=-1)
isinstance(m[1],nn.Flatten)

Flatten(start_dim=1, end_dim=-1)


True

In [34]:
model.linear_relu_stack[0].weight.dtype

torch.float32

In [25]:
print([x for x in model.named_modules()])
# for name,layer in model.named_modules():
#     print('='*6)
#     print('name:',name)
#     print('layer:',layer)

[('', net(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=2352, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)), ('flatten', Flatten(start_dim=1, end_dim=-1)), ('linear_relu_stack', Sequential(
  (0): Linear(in_features=2352, out_features=512, bias=True)
  (1): ReLU()
  (2): Linear(in_features=512, out_features=512, bias=True)
  (3): ReLU()
  (4): Linear(in_features=512, out_features=10, bias=True)
)), ('linear_relu_stack.0', Linear(in_features=2352, out_features=512, bias=True)), ('linear_relu_stack.1', ReLU()), ('linear_relu_stack.2', Linear(in_features=512, out_features=512, bias=True)), ('linear_relu_stack.3', ReLU()), ('linear_relu_stack.4', Linear(in_features=512, out_features=10, bias=True))]


In [26]:
print([x for x in model.children()])

[Flatten(start_dim=1, end_dim=-1), Sequential(
  (0): Linear(in_features=2352, out_features=512, bias=True)
  (1): ReLU()
  (2): Linear(in_features=512, out_features=512, bias=True)
  (3): ReLU()
  (4): Linear(in_features=512, out_features=10, bias=True)
)]


In [8]:
print([x for x in model.named_children()])

[('flatten', Flatten(start_dim=1, end_dim=-1)), ('linear_relu_stack', Sequential(
  (0): Linear(in_features=2352, out_features=512, bias=True)
  (1): ReLU()
  (2): Linear(in_features=512, out_features=512, bias=True)
  (3): ReLU()
  (4): Linear(in_features=512, out_features=10, bias=True)
))]


In [9]:
def setup_seed(seed):
    # 1. 设置pytorch的随机种子
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    # 2. 设置其他第3方库的随机变量
    np.random.seed(seed)
    random.seed(seed)
setup_seed(10101)

In [10]:
x = torch.rand(1,3,28,28).to(device)
logits = model(x)
print('logits.shape:',logits.shape)
prob = nn.Softmax(dim=1)(logits)
print('prob.shape:',prob.shape)
y_pred = prob.argmax(1)
print(f'predicted class:{y_pred.item()}')

logits.shape: torch.Size([1, 10])
prob.shape: torch.Size([1, 10])
predicted class:1


In [11]:
for name, param in model.named_parameters():
    print('name:',name,'size:',param.size(),'shape:',param.shape,'requires_grad:',param.requires_grad)
    # print(name,param.size(),param[:2],param.requires_grad)

name: linear_relu_stack.0.weight size: torch.Size([512, 2352]) shape: torch.Size([512, 2352]) requires_grad: True
name: linear_relu_stack.0.bias size: torch.Size([512]) shape: torch.Size([512]) requires_grad: True
name: linear_relu_stack.2.weight size: torch.Size([512, 512]) shape: torch.Size([512, 512]) requires_grad: True
name: linear_relu_stack.2.bias size: torch.Size([512]) shape: torch.Size([512]) requires_grad: True
name: linear_relu_stack.4.weight size: torch.Size([10, 512]) shape: torch.Size([10, 512]) requires_grad: True
name: linear_relu_stack.4.bias size: torch.Size([10]) shape: torch.Size([10]) requires_grad: True


In [12]:
print("hello world!!!")

hello world!!!


In [13]:
print("---------------模型参数冻结前---------------")
for name, param in model.named_parameters():
    print(f'name: {name}, requires_grad: {param.requires_grad}')
    
    
for param in model.parameters():
    param.requires_grad = False
print("---------------模型参数冻结后---------------")
# 验证参数是否被冻结
for name, param in model.named_parameters():
    print(f'name: {name}, requires_grad: {param.requires_grad}')

---------------模型参数冻结前---------------
name: linear_relu_stack.0.weight, requires_grad: True
name: linear_relu_stack.0.bias, requires_grad: True
name: linear_relu_stack.2.weight, requires_grad: True
name: linear_relu_stack.2.bias, requires_grad: True
name: linear_relu_stack.4.weight, requires_grad: True
name: linear_relu_stack.4.bias, requires_grad: True
---------------模型参数冻结后---------------
name: linear_relu_stack.0.weight, requires_grad: False
name: linear_relu_stack.0.bias, requires_grad: False
name: linear_relu_stack.2.weight, requires_grad: False
name: linear_relu_stack.2.bias, requires_grad: False
name: linear_relu_stack.4.weight, requires_grad: False
name: linear_relu_stack.4.bias, requires_grad: False


In [14]:
model.linear_relu_stack[0].weight

Parameter containing:
tensor([[ 0.0201,  0.0040,  0.0156,  ..., -0.0140, -0.0180, -0.0133],
        [ 0.0086, -0.0044, -0.0126,  ...,  0.0179,  0.0057, -0.0067],
        [ 0.0068, -0.0076,  0.0138,  ...,  0.0048,  0.0038, -0.0079],
        ...,
        [ 0.0097, -0.0012,  0.0141,  ..., -0.0135, -0.0185,  0.0027],
        [-0.0135,  0.0033, -0.0053,  ...,  0.0173,  0.0080, -0.0056],
        [ 0.0189, -0.0140,  0.0086,  ...,  0.0171,  0.0005,  0.0077]])

In [15]:
x = torch.ones((1,2),requires_grad=True) # 默认requires_grad=False
x.requires_grad

True

In [16]:
import torch
x = torch.ones(1,2)
print(x,'requires_grad=',x.requires_grad)
print("-="*6)
x = nn.Parameter(x)
print(x)
x.requires_grad

tensor([[1., 1.]]) requires_grad= False
-=-=-=-=-=-=
Parameter containing:
tensor([[1., 1.]], requires_grad=True)


True

In [17]:
import torch
inp = torch.eye(5, requires_grad=True)
out = (inp+1).pow(2)
print(out)
out.backward(torch.ones_like(inp), retain_graph=False)
inp.grad

tensor([[4., 1., 1., 1., 1.],
        [1., 4., 1., 1., 1.],
        [1., 1., 4., 1., 1.],
        [1., 1., 1., 4., 1.],
        [1., 1., 1., 1., 4.]], grad_fn=<PowBackward0>)


tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.],
        [2., 2., 2., 2., 4.]])

In [18]:
# https://zhuanlan.zhihu.com/p/515043918

In [19]:
torch.exp(torch.tensor(1.0))

tensor(2.7183)